# Preprocessing User Dataset
Pipeline lengkap: load → 10-core filter → filter item berdasarkan audio features → encoding → simpan.

In [2]:
import pandas as pd
import pickle
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

## 1. Load Data

In [3]:
df = pd.read_csv('data/original_data/lastfm_dataset.csv')
df = df.dropna(subset=['user_id', 'track_id', 'playcount'])

# Agregasi jika ada duplikat user-item
df = df.groupby(['user_id', 'track_id'], as_index=False)['playcount'].sum()

print(f"Total interaksi awal : {len(df)}")
print(f"Users                : {df['user_id'].nunique()}")
print(f"Items                : {df['track_id'].nunique()}")

Total interaksi awal : 730071
Users                : 21476
Items                : 6225


## 2. Filter 10-Core
Hanya pertahankan user dan item yang memiliki minimal 10 interaksi.

In [4]:
def filter_min_interactions(df, min_user=10, min_item=10, max_iter=10):
    df = df.copy()
    for _ in range(max_iter):
        prev_len = len(df)
        user_counts = df.groupby('user_id').size()
        df = df[df['user_id'].isin(user_counts[user_counts >= min_user].index)]
        item_counts = df.groupby('track_id').size()
        df = df[df['track_id'].isin(item_counts[item_counts >= min_item].index)]
        if len(df) == prev_len:
            break
    return df

df = filter_min_interactions(df, min_user=10, min_item=10)

print(f"Setelah 10-core filter:")
print(f"  Users      : {df['user_id'].nunique()}")
print(f"  Items      : {df['track_id'].nunique()}")
print(f"  Interaksi  : {len(df)}")

Setelah 10-core filter:
  Users      : 12044
  Items      : 4882
  Interaksi  : 687744


## 3. Filter Item Berdasarkan Audio Features
Hanya pertahankan item yang memiliki data audio features di `item_dataset.csv`,
sehingga semua item dijamin mendapat cluster assignment dari K-Means.
Setelah filter, 10-core dijalankan ulang karena beberapa user bisa kehilangan interaksi.

In [5]:
item_dataset = pd.read_csv('data/item_dataset.csv')
item_with_features = set(item_dataset['track_id'])

items_before = df['track_id'].nunique()
df = df[df['track_id'].isin(item_with_features)]

# Re-apply 10-core karena sebagian user mungkin kehilangan interaksi
df = filter_min_interactions(df, min_user=10, min_item=10)

items_after = df['track_id'].nunique()

print(f"Item sebelum filter audio features : {items_before}")
print(f"Item tanpa audio features (dihapus): {items_before - items_after}")
print(f"Item setelah filter                : {items_after}")
print(f"\nSetelah re-10-core:")
print(f"  Users      : {df['user_id'].nunique()}")
print(f"  Items      : {df['track_id'].nunique()}")
print(f"  Interaksi  : {len(df)}")

Item sebelum filter audio features : 4882
Item tanpa audio features (dihapus): 738
Item setelah filter                : 4144

Setelah re-10-core:
  Users      : 9529
  Items      : 4144
  Interaksi  : 561227


## 4. Implicit Feedback & Encoding

In [6]:
df['label'] = 1

user_enc = LabelEncoder()
item_enc = LabelEncoder()

df['user_id_enc'] = user_enc.fit_transform(df['user_id'])
df['item_id_enc'] = item_enc.fit_transform(df['track_id'])

print(f"Users      : {df['user_id_enc'].nunique()}")
print(f"Items      : {df['item_id_enc'].nunique()}")
print(f"Interaksi  : {len(df)}")
print(f"Max user_id_enc : {df['user_id_enc'].max()}")
print(f"Max item_id_enc : {df['item_id_enc'].max()}")
print(df.head())

Users      : 9529
Items      : 4144
Interaksi  : 561227
Max user_id_enc : 9528
Max item_id_enc : 4143
     user_id                track_id  playcount  label  user_id_enc  \
0  --mopsi--  05mAIVLkIWc2d1UBYZBCp8         40      1            0   
1  --mopsi--  086myS9r57YsLbJpU0TgK9         70      1            0   
2  --mopsi--  0V3wPSX9ygBnCm8psDIegu         45      1            0   
3  --mopsi--  0VjIjW4GlUZAMYd2vXMi3b         57      1            0   
4  --mopsi--  0cqRj7pUJDkTCEsJkx8snD         38      1            0   

   item_id_enc  
0           70  
1           91  
2          324  
3          333  
4          402  


## 5. LOO Splitting Data

In [9]:
## 5. LOO Split & Negative Sampling
import numpy as np
import random
import pandas as pd
from pathlib import Path
import pickle

print("Memulai proses LOO Split dan Negative Sampling. Harap tunggu sebentar (sekitar 10-20 detik)...")

# Set seed agar hasil split selalu konsisten dan bisa direproduksi
np.random.seed(42)
random.seed(42)

# Ambil semua ID item unik
all_items = set(df['item_id_enc'].unique())

# Kelompokkan semua item yang pernah didengar berdasarkan user
user_groups = df.groupby('user_id_enc')['item_id_enc'].apply(list).to_dict()

train_data = []
test_data = []

for u, items in user_groups.items():
    # --- 1. Leave-One-Out Split ---
    test_item = random.choice(items) # 1 item acak untuk testing
    train_items = [i for i in items if i != test_item] # Sisanya untuk training
    
    interacted_items = set(items)
    
    # Kandidat negatif adalah semua lagu di dataset DIKURANGI lagu yang pernah didengar user
    negative_candidates = list(all_items - interacted_items)
    
    # --- 2. Test Negative Sampling (1 Positif + 99 Negatif) ---
    test_negatives = random.sample(negative_candidates, 99)
    test_data.append([u, test_item, 1]) # Masukkan 1 label positif
    for neg in test_negatives:
        test_data.append([u, neg, 0])   # Masukkan 99 label negatif
        
    # --- 3. Train Negative Sampling (1 Positif + 4 Negatif per interaksi) ---
    for pos_item in train_items:
        train_data.append([u, pos_item, 1]) # Masukkan 1 label positif
        
        # Ambil 4 negatif acak baru untuk setiap interaksi
        train_negatives = random.sample(negative_candidates, 4)
        for neg in train_negatives:
            train_data.append([u, neg, 0])  # Masukkan 4 label negatif

# Konversi ke DataFrame
df_train = pd.DataFrame(train_data, columns=['user_id_enc', 'item_id_enc', 'label'])
df_test = pd.DataFrame(test_data, columns=['user_id_enc', 'item_id_enc', 'label'])

print(f"\nSelesai! Ukuran Data Train : {len(df_train)} baris")
print(f"Selesai! Ukuran Data Test  : {len(df_test)} baris")


Memulai proses LOO Split dan Negative Sampling. Harap tunggu sebentar (sekitar 10-20 detik)...

Selesai! Ukuran Data Train : 2758490 baris
Selesai! Ukuran Data Test  : 952900 baris


## 6. Simpan Dataset & Encoder

In [10]:
## 6. Simpan Dataset & Encoder
Path('data/pkl').mkdir(parents=True, exist_ok=True)

# Simpan semua versi dataset
df.to_csv('data/user_dataset_final.csv', index=False) # Data full original encoder
df_train.to_csv('data/train_dataset.csv', index=False) # Data train (termasuk 4 negatif)
df_test.to_csv('data/test_dataset.csv', index=False)   # Data test (termasuk 99 negatif)

print("\nDataset disimpan:")
print("1. data/user_dataset_final.csv")
print("2. data/train_dataset.csv")
print("3. data/test_dataset.csv")

# Simpan Encoder
with open('data/pkl/user_encoder.pkl', 'wb') as f:
    pickle.dump(user_enc, f)

with open('data/pkl/item_encoder.pkl', 'wb') as f:
    pickle.dump(item_enc, f)

print("\nEncoder disimpan:")
print("1. data/pkl/user_encoder.pkl")
print("2. data/pkl/item_encoder.pkl")


Dataset disimpan:
1. data/user_dataset_final.csv
2. data/train_dataset.csv
3. data/test_dataset.csv

Encoder disimpan:
1. data/pkl/user_encoder.pkl
2. data/pkl/item_encoder.pkl


---
### Membuat 4.144 Item dataset

In [3]:
import pandas as pd
from pathlib import Path

user_final = pd.read_csv("../data/user_dataset_final.csv")

item_final = (
    user_final[["track_id", "item_id_enc"]]
    .drop_duplicates()
    .sort_values("item_id_enc")
    .rename(columns={"item_id_enc": "item_id"})
)

print("Jumlah item final:", item_final["item_id"].nunique())
print("Range item_id:", item_final["item_id"].min(), "-", item_final["item_id"].max())

assert item_final["item_id"].nunique() == 4144
assert item_final["item_id"].min() == 0
assert item_final["item_id"].max() == 4143

Path("data").mkdir(exist_ok=True)
item_final.to_csv("../data/item_final_4144.csv", index=False)

print("Saved: ../data/item_final_4144.csv")

Jumlah item final: 4144
Range item_id: 0 - 4143
Saved: ../data/item_final_4144.csv
